In [1]:
import google.genai
import playwright
import dotenv
import sys

print("Python:", sys.executable)
print("Gemini SDK: OK")
print("Playwright: OK")
print("Dotenv: OK")

Python: C:\Users\manoj\myenv\Scripts\python.exe
Gemini SDK: OK
Playwright: OK
Dotenv: OK


In [3]:
!{sys.executable} -m playwright install chromium

In [4]:
import os
from getpass import getpass
from google import genai

api_key = getpass("Enter your NEW Gemini API key: ")

os.environ["GEMINI_API_KEY"] = api_key.strip()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

del api_key

print("Gemini client initialized!")


Enter your NEW Gemini API key:  ········


Gemini client initialized!


In [5]:
for model in ["gemini-3.5-flash-lite", "gemini-3.5-flash"]:
    try:
        print("Testing:", model)

        response = client.models.generate_content(
            model=model,
            contents="Reply only: API WORKING"
        )

        print(response.text)
        MODEL = model
        break

    except Exception as e:
        print(type(e).__name__, str(e))

Testing: gemini-3.5-flash-lite
API WORKING


In [6]:
import sys
import playwright

print("Python:", sys.executable)
print("Playwright:", playwright.__file__)

Python: C:\Users\manoj\myenv\Scripts\python.exe
Playwright: C:\Users\manoj\myenv\Lib\site-packages\playwright\__init__.py


In [7]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(result.stdout)

print("ERROR:")
print(result.stderr)

print("Exit code:", result.returncode)

OUTPUT:

ERROR:

Exit code: 0


In [8]:
import asyncio
from playwright.async_api import async_playwright

async def test_chromium():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        print("SUCCESS: Chromium launched!")

        page = await browser.new_page()
        await page.goto("https://www.amazon.in")

        print("Amazon title:", await page.title())

        await browser.close()

await test_chromium()

SUCCESS: Chromium launched!
Amazon title: 


In [9]:
from playwright.async_api import async_playwright

pw = await async_playwright().start()

browser = await pw.chromium.launch(
    headless=False,
    slow_mo=500
)

context = await browser.new_context()
page = await context.new_page()

await page.goto(
    "https://www.amazon.in",
    wait_until="domcontentloaded",
    timeout=60000
)

await page.wait_for_timeout(5000)

print("Page title:", await page.title())
print("Current URL:", page.url)
print("Browser is ready!")

Page title: 
Current URL: https://www.amazon.in/
Browser is ready!


In [10]:
await page.goto("https://www.amazon.in")

await page.locator("#twotabsearchtextbox").wait_for(
    state="visible",
    timeout=30000
)

await page.locator("#twotabsearchtextbox").fill(
    "wireless mouse under 1000"
)

await page.locator("#nav-search-submit-button").click()

await page.wait_for_timeout(5000)

print("Search page:", await page.title())

Search page: Amazon.in : wireless mouse under 1000


In [ ]:
async def open_amazon() -> str:
    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )
    return "Amazon India opened successfully."


async def search_product(keyword: str) -> str:
    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.locator("#twotabsearchtextbox").fill(keyword)
    await page.locator("#nav-search-submit-button").click()

    await page.wait_for_selector(
        "[data-component-type='s-search-result']",
        timeout=30000
    )

    return f"Search completed for {keyword}."


async def get_top_products(limit: int = 5) -> str:
    cards = page.locator(
        "[data-component-type='s-search-result']"
    )

    results = []

    for i in range(min(limit, await cards.count())):
        card = cards.nth(i)

        title_loc = card.locator("h2 span").first
        price_loc = card.locator(".a-price .a-offscreen").first

        title = (
            await title_loc.inner_text()
            if await title_loc.count()
            else "Title unavailable"
        )

        price = (
            await price_loc.inner_text()
            if await price_loc.count()
            else "Price unavailable"
        )

        results.append(f"{i+1}. {title.strip()} | {price.strip()}")

    return "\n".join(results) if results else "No products found."

print("All three Amazon tools are ready!")


In [11]:
await search_product("wireless mouse under 1000")

products = await get_top_products(5)

print(products)

NameError: name 'search_product' is not defined

In [12]:
from google.genai import types

tools = [
    open_amazon,
    search_product,
    get_top_products
]

chat = client.aio.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        tools=tools,
        temperature=0
    )
)

goal = """
Open Amazon India.
Search for wireless mouse under 1000.
Extract the top 5 products.
Compare the available prices.
Recommend the best value product.

Only use information actually collected from Amazon.
Do not invent prices or specifications.
"""

response = await chat.send_message(goal)

print(response.text)

NameError: name 'open_amazon' is not defined

In [13]:

async def open_amazon() -> str:
    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )
    return "Amazon India opened successfully."


async def search_product(keyword: str) -> str:
    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.locator("#twotabsearchtextbox").fill(keyword)
    await page.locator("#nav-search-submit-button").click()

    await page.wait_for_selector(
        "[data-component-type='s-search-result']",
        timeout=30000
    )

    return f"Search completed for {keyword}."


async def get_top_products(limit: int = 5) -> str:
    cards = page.locator(
        "[data-component-type='s-search-result']"
    )

    results = []

    for i in range(min(limit, await cards.count())):
        card = cards.nth(i)

        title_loc = card.locator("h2 span").first
        price_loc = card.locator(".a-price .a-offscreen").first

        title = (
            await title_loc.inner_text()
            if await title_loc.count() > 0
            else "Title unavailable"
        )

        price = (
            await price_loc.inner_text()
            if await price_loc.count() > 0
            else "Price unavailable"
        )

        results.append(
            f"{i+1}. {title.strip()} | {price.strip()}"
        )

    return "\n".join(results) if results else "No products found."


print("SUCCESS: All three functions are defined!")


SUCCESS: All three functions are defined!


In [14]:
print(open_amazon)
print(search_product)
print(get_top_products)

<function open_amazon at 0x00000164A246F740>
<function search_product at 0x00000164A246F7F0>
<function get_top_products at 0x00000164A246F950>


In [15]:
await search_product("wireless mouse under 1000")

products = await get_top_products(5)

print(products)

1. Title unavailable | Price unavailable


In [16]:
print("CURRENT URL:", page.url)
print("PAGE TITLE:", await page.title())

cards = page.locator("[data-component-type='s-search-result']")

print("PRODUCT CARDS FOUND:", await cards.count())

if await cards.count() > 0:
    first_card = cards.nth(0)

    print("\n========== FIRST PRODUCT ==========")
    print(await first_card.inner_text())

    print("\n========== HTML STRUCTURE ==========")
    print((await first_card.inner_html())[:5000])

else:
    print("\nNo product cards found.")
    print((await page.locator("body").inner_text())[:2000])

CURRENT URL: https://www.amazon.in/s?k=wireless+mouse+under+1000&crid=3TSBA5GMMPJT4&sprefix=wireless+mouse+under+1000%2Caps%2C337&ref=nb_sb_noss_1
PAGE TITLE: Amazon.in : wireless mouse under 1000
PRODUCT CARDS FOUND: 22

========== FIRST PRODUCT ==========
Sponsored 
Amkette Hush Pro Epic M Rechargeable Wireless Mouse/Bluetooth Mouse | 3 Device Connectivity (2xBT&2.4G) with Top Mounted Device Switch Button, Silent Switches, Ergonomic Design, Upto 3200 DPI (White)
Top Reviewed for Connectivity
4.1
4.1 out of 5 stars
 (1.3K)
300+ bought in past month
Limited time deal
Price, product page
₹699
₹699 
M.R.P: ₹1,199
M.R.P: 
₹1,199
₹1,199 (42% off)
Up to 5% back with Amazon Pay ICICI card
Up to 5% back with Amazon Pay ICI...
FREE delivery Tue, 1 Dec
Add to cart

========== HTML STRUCTURE ==========
<div class="sg-col-inner"><div cel_widget_id="MAIN-SEARCH_RESULTS-4" class="s-widget-container s-spacing-small s-widget-container-height-small celwidget slot=MAIN template=SEARCH_RESULTS widgetId=

In [17]:
async def get_top_products(limit=5):

    cards = page.locator(
        "[data-component-type='s-search-result']"
    )

    results = []

    for i in range(min(limit, await cards.count())):

        card = cards.nth(i)

        # Try multiple title selectors
        title = "Title unavailable"

        for selector in [
            "h2 a span",
            "h2 span",
            "h2"
        ]:
            locator = card.locator(selector).first

            if await locator.count() > 0:
                value = (await locator.inner_text()).strip()

                if value:
                    title = value
                    break

        # Try multiple price selectors
        price = "Price unavailable"

        for selector in [
            ".a-price .a-offscreen",
            ".a-price-whole",
            "[class*='priceToPay']"
        ]:
            locator = card.locator(selector).first

            if await locator.count() > 0:
                value = (await locator.inner_text()).strip()

                if value:
                    price = value
                    break

        results.append({
            "product": title,
            "price": price
        })

    for i, item in enumerate(results, 1):
        print(f"{i}. {item['product']} | {item['price']}")

    return results

In [18]:
await search_product("wireless mouse under 1000")

products = await get_top_products(5)

1. Amkette Hush Pro Epic M Rechargeable Wireless Mouse/Bluetooth Mouse | 3 Device Connectivity (2xBT&2.4G) with Top Mounted Device Switch Button, Silent Switches, Ergonomic Design, Upto 3200 DPI (White) | ₹699
2. EvoFox Blaze 2 Wired Gaming Mouse | High Performance 1000Hz Polling Rate, Upto 12,800 DPI, RGB Lighting, 8 Programmable Buttons, Software Support with On-Board Memory, Durable Huano Switches | ₹649
3. EvoFox Shadow X Pro Wireless Gaming Mouse Tri-Mode Rechargeable | BT, USB 2.4G & Wired, S203 Sensor, 1000Hz Polling Rate, 6400 DPI, RGB Lighting, 6 Programmable Buttons, Software Support | ₹999
4. Portronics Toad Fold Wireless Bluetooth Mouse with Dual Wireless (BT + 2.4 GHz), Foldable Mouse, Rechargeable Battery, Multi-Device Pair, Up to 3200 DPI Sensitivity (Black) | ₹899
5. Acer Prism Dual Mode Wireless Mouse I 2.4G + Bluetooth I Silent Click I 3200 DPI (800-1200-2400-3200) I Rechargeable 500mAh I RGB LED I 6 Buttons I Lightweight 87g I Ergonomic Design for Laptop PC Mac | ₹59